# Advanced Hybrid Search: Score Fusion, Weight Tuning, and Evaluation

这份 notebook 是对 `2-1-RetrievalAugmentedGeneration/Vector_Database.ipynb` 里 `hybrid search` 部分的深入补全。

前面的内容已经让你知道：

- hybrid search = 稀疏检索 + 稠密检索的结合

但还没有真正展开几个更关键的问题：

- hybrid score 到底怎么融合
- dense / sparse 的权重怎么调
- 不同 query 类型下，哪种检索更占优势
- 怎么评测不同 hybrid 配置

这一节就专门把这四个问题拆开讲。


## 这一节的技术在做什么

`Hybrid Search` 的核心目标不是“把两个分数加一下”这么简单，而是解决一个更现实的问题：

- 有些问题靠关键词匹配更强
- 有些问题靠语义相似度更强
- 真正稳定的检索系统，往往要把两者结合

在这份 notebook 里，你会看到四个核心主题：

1. `hybrid score 融合策略`
2. `dense / sparse 权重调优`
3. `不同 query 类型下的召回差异`
4. `评测不同 hybrid 配置`

这意味着我们不只看“能不能查到”，而是进一步看：

- 为什么查到
- 是稀疏通道起了主要作用，还是稠密通道起了主要作用
- 不同类型的问题，应该如何调 hybrid 的权重


## Imports


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 1. Load the same restaurant knowledge used in the existing RAG lessons


In [ ]:
shared_data_dir = Path("/Users/a1-6/Desktop/AIAgent/code/2-1-RetrievalAugmentedGeneration/shared_data")

docs = []
for path in [
    shared_data_dir / "menu_and_ordering.md",
    shared_data_dir / "delivery_and_pickup.md",
]:
    docs.append(
        {
            "doc_id": path.stem,
            "text": path.read_text(encoding="utf-8"),
        }
    )

# 为了让 hybrid 的差异更明显，再把原文切成更小的“知识块”。
# 这样每个 chunk 都更像真实 RAG 系统里的一个候选片段。
chunked_docs = []
for doc in docs:
    paragraphs = [p.strip() for p in doc["text"].split("\n\n") if p.strip()]
    for index, paragraph in enumerate(paragraphs, start=1):
        chunked_docs.append(
            {
                "chunk_id": f"{doc['doc_id']}_{index}",
                "source_doc": doc["doc_id"],
                "text": paragraph,
            }
        )

len(chunked_docs), chunked_docs[0]


## 2. Build sparse and dense retrievers separately


In [ ]:
def find_local_bge_snapshot() -> Path:
    snapshot_root = (
        Path("/Users/a1-6/Desktop/AIAgent/models")
        / "models--BAAI--bge-small-en-v1.5"
        / "snapshots"
    )
    snapshots = sorted(
        path for path in snapshot_root.iterdir()
        if path.is_dir() and (path / "config.json").exists()
    )
    if not snapshots:
        raise FileNotFoundError(f"No valid BGE snapshot found under {snapshot_root}")
    return snapshots[0]


texts = [item["text"] for item in chunked_docs]

# sparse 检索这边用 TF-IDF 来模拟关键词/词项检索。
# 它不懂真正的语义，但对“精确字面词”非常敏感。
tfidf = TfidfVectorizer(stop_words="english")
sparse_matrix = tfidf.fit_transform(texts)

# dense 检索这边用本地 BGE embedding。
# 它更像向量数据库背后的语义召回能力。
dense_model = SentenceTransformer(str(find_local_bge_snapshot()))
dense_embeddings = dense_model.encode(texts, normalize_embeddings=True)


def sparse_scores(query: str) -> np.ndarray:
    query_vec = tfidf.transform([query])
    return cosine_similarity(query_vec, sparse_matrix)[0]


def dense_scores(query: str) -> np.ndarray:
    query_vec = dense_model.encode([query], normalize_embeddings=True)
    return cosine_similarity(query_vec, dense_embeddings)[0]


## 3. Define hybrid score fusion


In [ ]:
def min_max_normalize(scores: np.ndarray) -> np.ndarray:
    # 不同通道出来的分数尺度不一定一致，所以融合前先归一化。
    # 这是最常见、也最容易理解的一种 hybrid score 融合预处理方式。
    scores = np.asarray(scores, dtype=float)
    if np.allclose(scores.max(), scores.min()):
        return np.zeros_like(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())


def hybrid_scores(query: str, dense_weight: float = 0.5, sparse_weight: float = 0.5) -> pd.DataFrame:
    # 这里演示的是“加权融合”策略：
    # hybrid_score = dense_weight * dense_score + sparse_weight * sparse_score
    #
    # 现实里也有 Reciprocal Rank Fusion、learned fusion 等更复杂的方法，
    # 但加权融合最适合用来理解 hybrid search 的底层逻辑。
    sparse = sparse_scores(query)
    dense = dense_scores(query)

    sparse_norm = min_max_normalize(sparse)
    dense_norm = min_max_normalize(dense)
    hybrid = dense_weight * dense_norm + sparse_weight * sparse_norm

    rows = []
    for item, sparse_value, dense_value, hybrid_value in zip(chunked_docs, sparse_norm, dense_norm, hybrid):
        rows.append(
            {
                "chunk_id": item["chunk_id"],
                "source_doc": item["source_doc"],
                "sparse_score": float(sparse_value),
                "dense_score": float(dense_value),
                "hybrid_score": float(hybrid_value),
                "text": item["text"],
            }
        )

    return pd.DataFrame(rows).sort_values("hybrid_score", ascending=False).reset_index(drop=True)


## 4. Compare query types


In [ ]:
# 这组 query 特意分成不同风格，方便你观察 sparse / dense 各自更擅长什么。
query_examples = [
    {
        "query_type": "exact_keyword",
        "query": "What is the pickup policy?",
    },
    {
        "query_type": "semantic_paraphrase",
        "query": "How does the restaurant handle customers collecting food themselves?",
    },
    {
        "query_type": "menu_lookup",
        "query": "Which items are on the menu?",
    },
    {
        "query_type": "intent_style",
        "query": "I want to know how ordering and food collection work.",
    },
]

for example in query_examples:
    query = example["query"]
    df = hybrid_scores(query, dense_weight=0.5, sparse_weight=0.5)
    print(f"Query type: {example['query_type']}")
    print(f"Query: {query}")
    print(df[["chunk_id", "sparse_score", "dense_score", "hybrid_score"]].head(3))
    print("-" * 100)


## 5. Tune dense / sparse weights


In [ ]:
# 这里构造一个很小的“带标签检索评测集”。
# 它的目标不是做工业级 benchmark，而是让你看到：
# 同一批 query，在不同 hybrid 配置下，命中情况会变。
eval_examples = [
    {
        "query": "What is the pickup policy?",
        "query_type": "exact_keyword",
        "expected_source_doc": "delivery_and_pickup",
    },
    {
        "query": "How does the restaurant handle customers collecting food themselves?",
        "query_type": "semantic_paraphrase",
        "expected_source_doc": "delivery_and_pickup",
    },
    {
        "query": "Which items are available to order?",
        "query_type": "semantic_menu",
        "expected_source_doc": "menu_and_ordering",
    },
    {
        "query": "Tell me about ordering rules and the food menu.",
        "query_type": "mixed_intent",
        "expected_source_doc": "menu_and_ordering",
    },
]


def top1_hit_rate(dense_weight: float, sparse_weight: float) -> pd.DataFrame:
    rows = []
    for item in eval_examples:
        ranking = hybrid_scores(item["query"], dense_weight=dense_weight, sparse_weight=sparse_weight)
        top_row = ranking.iloc[0]
        hit = top_row["source_doc"] == item["expected_source_doc"]
        rows.append(
            {
                "query": item["query"],
                "query_type": item["query_type"],
                "expected_source_doc": item["expected_source_doc"],
                "top1_source_doc": top_row["source_doc"],
                "hit": hit,
            }
        )
    return pd.DataFrame(rows)


weight_configs = [
    {"dense_weight": 0.2, "sparse_weight": 0.8},
    {"dense_weight": 0.5, "sparse_weight": 0.5},
    {"dense_weight": 0.8, "sparse_weight": 0.2},
]

summary_rows = []
for config in weight_configs:
    result_df = top1_hit_rate(**config)
    summary_rows.append(
        {
            **config,
            "top1_accuracy": result_df["hit"].mean(),
        }
    )

pd.DataFrame(summary_rows)


## 6. Inspect per-query recall differences across configurations


In [ ]:
detailed_rows = []
for config in weight_configs:
    result_df = top1_hit_rate(**config)
    grouped = result_df.groupby("query_type")["hit"].mean().reset_index()
    for _, row in grouped.iterrows():
        detailed_rows.append(
            {
                "dense_weight": config["dense_weight"],
                "sparse_weight": config["sparse_weight"],
                "query_type": row["query_type"],
                "top1_accuracy": row["hit"],
            }
        )

pd.DataFrame(detailed_rows).sort_values(["query_type", "dense_weight"])
